# Estudo de Robustez Temporal -- XGBoost para Classificacao (SLCE3 + Indicadores)

Replica o experimento `xgboost/dataset_indicadores/xgboost_slce3` substituindo:

- `train_test_split(shuffle=True, stratify=y)` por **holdout temporal** (ultimos 20% das amostras como teste)
- `StratifiedKFold(shuffle=True)` por `TimeSeriesSplit(n_splits=5)`

Mantem-se a calibracao do threshold via curva Precision-Recall sobre as predicoes Out-Of-Fold do `Dev`.

Objetivo: quantificar o vies introduzido pela validacao aleatoria estratificada na tarefa de classificacao binaria direcional (Cap. 4, Secao 4.6 da monografia).

In [1]:
import warnings
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

OUTPUT_DIR = Path('./')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

COMPANY = 'SLCE3'
DATASET_LABEL = 'indicadores_timeseries'
company_lower = COMPANY.lower()

df = pd.read_csv('../../datasets/datasets_indicadores/classificacao/SLCE3_tratado.csv', index_col=0, parse_dates=True)
df = df.sort_index()

print(f'Dataset {COMPANY} (clf indicadores): {df.shape}')
print(f'Periodo: {df.index.min().date()} a {df.index.max().date()}')

Dataset SLCE3 (clf indicadores): (1633, 24)
Periodo: 2018-02-15 a 2024-11-11


In [2]:
# Features = todas as colunas exceto os 4 targets binarios e os 4 close futuros (se presentes)
target_cols = ['target_3d', 'target_7d', 'target_15d', 'target_30d']
future_cols = ['Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut']
feat_cols = [c for c in df.columns if c not in target_cols + future_cols]

X = df[feat_cols].copy()
targets = {p: df[f'target_{p}'] for p in ['3d', '7d', '15d', '30d']}

print(f'Total de features: {len(feat_cols)}')
for p, y in targets.items():
    pos = (y == 1).sum()
    print(f'  Horizonte {p}: {len(y)} amostras | positivos = {pos} ({pos/len(y):.1%})')

Total de features: 20
  Horizonte 3d: 1633 amostras | positivos = 823 (50.4%)
  Horizonte 7d: 1633 amostras | positivos = 858 (52.5%)
  Horizonte 15d: 1633 amostras | positivos = 875 (53.6%)
  Horizonte 30d: 1633 amostras | positivos = 896 (54.9%)


In [3]:
param_dist = {
    'model__n_estimators': [100, 200, 300, 400],
    'model__max_depth': [3, 4, 5, 6, 8],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'model__min_child_weight': [1, 3, 5],
    'model__reg_lambda': [0.5, 1.0, 2.0],
    'model__gamma': [0.0, 0.1, 0.5],
}
cv_strategy = TimeSeriesSplit(n_splits=5)
print('CV strategy:', cv_strategy)

CV strategy: TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)


In [4]:
results = {}
for period, y in targets.items():
    print(f'\n========== Horizonte {period.upper()} ==========')
    mask = ~y.isnull()
    X_clean = X.loc[mask].copy()
    y_clean = y.loc[mask].astype(int).copy()

    n_total = len(X_clean)
    n_test = int(round(0.2 * n_total))
    X_dev = X_clean.iloc[: n_total - n_test].copy()
    y_dev = y_clean.iloc[: n_total - n_test].copy()
    X_test = X_clean.iloc[n_total - n_test :].copy()
    y_test = y_clean.iloc[n_total - n_test :].copy()

    print(f'  Dev: {X_dev.index.min().date()} a {X_dev.index.max().date()} ({len(X_dev)} amostras, {y_dev.mean():.1%} positivos)')
    print(f'  Teste: {X_test.index.min().date()} a {X_test.index.max().date()} ({len(X_test)} amostras, {y_test.mean():.1%} positivos)')

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', XGBClassifier(
            objective='binary:logistic', eval_metric='logloss',
            random_state=42, n_jobs=-1, verbosity=0,
        )),
    ])

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_dist,
        n_iter=24,
        scoring='f1',
        cv=cv_strategy,
        n_jobs=-1,
        random_state=42,
        verbose=0,
    )
    search.fit(X_dev, y_dev)
    best_pipeline = cast(Pipeline, search.best_estimator_)
    cv_f1_best = search.best_score_

    # OOF temporal manual para calibrar threshold (TimeSeriesSplit nao cobre todas as amostras)
    oof_proba = np.full(len(y_dev), np.nan)
    for tr_idx, val_idx in cv_strategy.split(X_dev):
        clone_pipe = clone(best_pipeline)
        clone_pipe.fit(X_dev.iloc[tr_idx], y_dev.iloc[tr_idx])
        oof_proba[val_idx] = clone_pipe.predict_proba(X_dev.iloc[val_idx])[:, 1]
    mask_oof = ~np.isnan(oof_proba)
    y_dev_oof = y_dev.iloc[mask_oof].values
    p_oof = oof_proba[mask_oof]
    precisions, recalls, thresholds = precision_recall_curve(y_dev_oof, p_oof)
    f1s = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-12)
    best_idx = int(np.argmax(f1s))
    threshold_otimo = float(thresholds[best_idx])
    oof_f1 = float(f1s[best_idx])

    # Refit + holdout
    best_pipeline.fit(X_dev, y_dev)
    test_proba = best_pipeline.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= threshold_otimo).astype(int)

    test_acc = accuracy_score(y_test, test_pred)
    test_f1_alta = f1_score(y_test, test_pred, pos_label=1, zero_division=0)
    test_f1_weighted = f1_score(y_test, test_pred, average='weighted', zero_division=0)
    test_auc = roc_auc_score(y_test, test_proba) if len(np.unique(y_test)) > 1 else float('nan')

    print(f'  CV F1 (best):       {cv_f1_best:.4f}')
    print(f'  Threshold otimo:     {threshold_otimo:.4f}')
    print(f'  OOF F1 no threshold: {oof_f1:.4f}')
    print(f'  Test Acc:            {test_acc:.4f}')
    print(f'  Test F1 (Alta):      {test_f1_alta:.4f}')
    print(f'  Test F1 (weighted):  {test_f1_weighted:.4f}')
    print(f'  Test AUC-ROC:        {test_auc:.4f}')

    results[period] = {
        'cv_f1': cv_f1_best,
        'threshold': threshold_otimo,
        'oof_f1': oof_f1,
        'test_acc': test_acc,
        'test_f1_alta': test_f1_alta,
        'test_f1_weighted': test_f1_weighted,
        'test_auc': test_auc,
    }


========== Horizonte 3D ==========
  Dev: 2018-02-15 a 2023-07-13 (1306 amostras, 51.7% positivos)
  Teste: 2023-07-14 a 2024-11-11 (327 amostras, 45.3% positivos)


  CV F1 (best):       0.4644
  Threshold otimo:     0.0162
  OOF F1 no threshold: 0.6813
  Test Acc:            0.4557
  Test F1 (Alta):      0.6245
  Test F1 (weighted):  0.2887
  Test AUC-ROC:        0.5806

========== Horizonte 7D ==========
  Dev: 2018-02-15 a 2023-07-13 (1306 amostras, 54.5% positivos)
  Teste: 2023-07-14 a 2024-11-11 (327 amostras, 44.6% positivos)


  CV F1 (best):       0.4821
  Threshold otimo:     0.0122
  OOF F1 no threshold: 0.6972
  Test Acc:            0.4465
  Test F1 (Alta):      0.6173
  Test F1 (weighted):  0.2756
  Test AUC-ROC:        0.6274

========== Horizonte 15D ==========
  Dev: 2018-02-15 a 2023-07-13 (1306 amostras, 56.0% positivos)
  Teste: 2023-07-14 a 2024-11-11 (327 amostras, 44.0% positivos)


  CV F1 (best):       0.5410
  Threshold otimo:     0.0089
  OOF F1 no threshold: 0.7134
  Test Acc:            0.4434
  Test F1 (Alta):      0.6043
  Test F1 (weighted):  0.3008
  Test AUC-ROC:        0.5597

========== Horizonte 30D ==========
  Dev: 2018-02-15 a 2023-07-13 (1306 amostras, 58.3% positivos)
  Teste: 2023-07-14 a 2024-11-11 (327 amostras, 41.3% positivos)


  CV F1 (best):       0.6910
  Threshold otimo:     0.0383
  OOF F1 no threshold: 0.7344
  Test Acc:            0.4434
  Test F1 (Alta):      0.5160
  Test F1 (weighted):  0.4158
  Test AUC-ROC:        0.6017


In [5]:
tabela = pd.DataFrame({
    'Horizonte': ['3 dias', '7 dias', '15 dias', '30 dias'],
    'CV_F1_Best': [results[p]['cv_f1'] for p in ['3d', '7d', '15d', '30d']],
    'Threshold_Otimo': [results[p]['threshold'] for p in ['3d', '7d', '15d', '30d']],
    'OOF_F1_Threshold': [results[p]['oof_f1'] for p in ['3d', '7d', '15d', '30d']],
    'Teste_Accuracy': [results[p]['test_acc'] for p in ['3d', '7d', '15d', '30d']],
    'Teste_F1_Weighted': [results[p]['test_f1_weighted'] for p in ['3d', '7d', '15d', '30d']],
    'Teste_F1_Classe_Alta': [results[p]['test_f1_alta'] for p in ['3d', '7d', '15d', '30d']],
    'Teste_AUC_ROC': [results[p]['test_auc'] for p in ['3d', '7d', '15d', '30d']],
})
print('=== Resultados XGBoost (Classificacao) -- SLCE3 -- TimeSeriesSplit ===')
print(tabela.to_string(index=False))
tabela.to_csv(OUTPUT_DIR / f'metricas_{company_lower}_{DATASET_LABEL}.csv', index=False)
print(f'\nSalvo: metricas_{company_lower}_{DATASET_LABEL}.csv')

=== Resultados XGBoost (Classificacao) -- SLCE3 -- TimeSeriesSplit ===
Horizonte  CV_F1_Best  Threshold_Otimo  OOF_F1_Threshold  Teste_Accuracy  Teste_F1_Weighted  Teste_F1_Classe_Alta  Teste_AUC_ROC
   3 dias    0.464439         0.016216          0.681265        0.455657           0.288718              0.624473       0.580590
   7 dias    0.482135         0.012206          0.697170        0.446483           0.275630              0.617336       0.627412
  15 dias    0.540971         0.008864          0.713353        0.443425           0.300751              0.604348       0.559654
  30 dias    0.690976         0.038261          0.734375        0.443425           0.415769              0.515957       0.601698

Salvo: metricas_slce3_indicadores_timeseries.csv
